# 0.6 Manifolds — 互動版（可拖曳旋轉）

這個 notebook 用 **ipympl**（`%matplotlib widget`）把 3D 圖變成可**拖曳旋轉 / 縮放**
的互動畫布，並用滑桿即時改變球面上的點。

**前置**（需要 `interactive` extra）：

```bash
uv sync --extra interactive
uv run jupyter lab
```

> 靜態（不需互動）版本請看 `notebooks/02_manifolds.ipynb`；本檔的圖只在
> 有 kernel 的 JupyterLab 中才會互動，因此輸出不預先內嵌。


In [ ]:
%matplotlib widget
import numpy as np

from scw.manifold import EuclideanSpace, Sphere
from scw.math_core import Vector
from scw.visualization import new_3d_figure, plot_vectors_3d

## 1. 平坦空間：切空間與 metric


In [ ]:
plane = EuclideanSpace(3, name="E")
p = (1.0, 2.0, 3.0)
g = plane.metric(p)
u = plane.tangent_vector([1.0, 0.0, 0.0], point=p)
print("T_pM =", plane.tangent_space(p))
print("G =", g.components.tolist(), " |u| =", g.norm(u))
print("sharp(flat(u)) = u ?", np.allclose(g.sharp(g.flat(u)).data, u.data))

## 2. 可拖曳的球面 + 切基底

把 `(theta, phi)` 座標嵌回 R^3，畫出 `e_theta`、`e_phi`。
**用滑鼠左鍵拖曳可旋轉，滾輪縮放。**


In [ ]:
R = 2.0


def embed(th, ph):
    return np.array(
        [R * np.sin(th) * np.cos(ph), R * np.sin(th) * np.sin(ph), R * np.cos(th)]
    )


def tangent_basis(th, ph):
    e_th = np.array(
        [R * np.cos(th) * np.cos(ph), R * np.cos(th) * np.sin(ph), -R * np.sin(th)]
    )
    e_ph = np.array([-R * np.sin(th) * np.sin(ph), R * np.sin(th) * np.cos(ph), 0.0])
    return e_th, e_ph


fig, ax = new_3d_figure(figsize=(6, 6))
TH, PH = np.meshgrid(np.linspace(0, np.pi, 80), np.linspace(0, 2 * np.pi, 80))
ax.plot_surface(
    R * np.sin(TH) * np.cos(PH),
    R * np.sin(TH) * np.sin(PH),
    R * np.cos(TH),
    alpha=0.18,
    color="tab:blue",
    linewidth=0,
)
th0, ph0 = np.pi / 3, np.pi / 6
pt = embed(th0, ph0)
e_th, e_ph = tangent_basis(th0, ph0)
ax.scatter(*pt, color="k", s=30, label="p")
plot_vectors_3d(
    [Vector(e_th), Vector(e_ph)],
    origins=[pt, pt],
    colors=["tab:red", "tab:green"],
    labels=["e_theta", "e_phi"],
    ax=ax,
)
ax.set_title("S^2: drag to rotate; g = diag(r^2, r^2 sin^2 theta)")
ax.legend(loc="upper right")
fig

## 3. 用滑桿移動球面上的點

拖動 `theta` / `phi` 滑桿，切基底與 metric 會即時更新。
注意 `|e_phi| = r sin(theta)` 在赤道（theta = pi/2）最大、越靠兩極越小。


In [ ]:
import ipywidgets as widgets

sphere = Sphere(radius=R)


def update(theta=np.pi / 3, phi=np.pi / 6):
    ax.clear()
    ax.plot_surface(
        R * np.sin(TH) * np.cos(PH),
        R * np.sin(TH) * np.sin(PH),
        R * np.cos(TH),
        alpha=0.18,
        color="tab:blue",
        linewidth=0,
    )
    pt = embed(theta, phi)
    e_th, e_ph = tangent_basis(theta, phi)
    ax.scatter(*pt, color="k", s=30)
    plot_vectors_3d(
        [Vector(e_th), Vector(e_ph)],
        origins=[pt, pt],
        colors=["tab:red", "tab:green"],
        labels=["e_theta", "e_phi"],
        ax=ax,
    )
    if 0.0 < theta < np.pi:
        m = sphere.metric((theta, phi))
        len_th = m.norm(sphere.tangent_vector([1.0, 0.0], point=(theta, phi)))
        len_ph = m.norm(sphere.tangent_vector([0.0, 1.0], point=(theta, phi)))
    else:
        len_th = len_ph = float("nan")  # pole: chart is singular
    ax.set_title(f"|e_theta| = {len_th:.3f}   |e_phi| = {len_ph:.3f}")
    ax.legend(loc="upper right")


widgets.interact(
    update,
    theta=widgets.FloatSlider(value=np.pi / 3, min=0.05, max=np.pi - 0.05, step=0.01),
    phi=widgets.FloatSlider(value=np.pi / 6, min=0.0, max=2 * np.pi, step=0.01),
)

## 4. 換你玩

- 把 `theta` 拉到靠近 0（極點），看 `|e_phi| = r sin(theta)` 如何趨近 0。
- 改 `R`（球半徑），看兩個切向量長度等比例放大。
- 在 3D 圖上按右鍵拖曳可平移、滾輪縮放。
